In [181]:
import pandas as pd
import numpy as np
import seaborn as sns
import csv
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.utils import resample

TARGET = 'target'
ID_COL = 'reco_id_curr'
RANDOM_STATE = 42

def split_xy(frame):
    y = frame[TARGET].astype(int)
    X = frame.drop(columns=[TARGET, ID_COL], errors='ignore')
    return X, y

def gini(y_true, y_pred):
    auc = roc_auc_score(y_true, y_pred)
    return 2 * auc  - 1

In [182]:
df = pd.read_csv('Data/train.csv')

In [183]:
data = df.copy()

In [184]:
# Замена аномалии в данных. В колонке 47104 строк со значением 365243
data['days_employed'].replace(365243, float('nan'))
pass

In [185]:
# Удаление признаков с > 60% пропусков
empty_columns = []
for column in data.columns:
    if (data[column].isnull().sum() / data.shape[0] > 0.6):
        empty_columns.append(column)
data = data.drop(empty_columns, axis=1)

#### Было 122 колонки стало 105. Удалено 17 (колонки с пропусками > 60%)

In [186]:
# Замена пропусков в строках медианой для численных и модой для категориальный
data = data.fillna(data.median(numeric_only=True))
for column in data.columns:
    data[column] = data[column].fillna(data[column].mode()[0])

In [187]:
# Удаление признаков, где одно значение встреается > 99%
const_columns = []
for column in data.columns:
    values_in_column = data[column].value_counts()
    if (values_in_column.max() / data.shape[0] > 0.99):
        const_columns.append(column)
data = data.drop(const_columns, axis=1)

#### Было 105 столбцов стало 83. Удалены околоконстантные признаки

In [188]:
# Удаление аномальный объектов (25 и 75 перцентили)

# Находим аномальные колонки
anomaly_columns = []
for column in (data.select_dtypes(include=['float64', 'int64'])).columns:
    data_column = data[column]
    q1 = data_column.quantile(0.25)
    q3 = data_column.quantile(0.75)
    iqr = q3 - q1
    cnt_anomaly = (data_column < q1 - 1.5 * iqr).sum() + (data_column > q3 + 1.5 * iqr).sum()
    procent_of_anomaly = cnt_anomaly / data.shape[0]
    if (0 < procent_of_anomaly < 0.01):
        anomaly_columns.append(column)

# Берём id всех аномалий
anomaly_id = set()
for column in anomaly_columns:
    data_column = data[column]
    q1 = data_column.quantile(0.25)
    q3 = data_column.quantile(0.75)
    iqr = q3 - q1
    anomaly_rows = data[(data[column] > q3 + 1.5 * iqr) | (data[column] < q1 - 1.5 * iqr)]
    anomaly_id.update(anomaly_rows['reco_id_curr'])

# Удаляем аномалии по id
anomaly_id = list(anomaly_id)
data = data[~data['reco_id_curr'].isin(anomaly_id)]

#### Строк было 261384 стало 258546

# Итог обработки данных
#### Колонок было 122 стало 83
#### Строк было 261384 стало 258546

# Разбиение данных
### Train / val / test / late
| Выборка | Доля | Зачем |
|---------|------|--------|
| train | 70% | Обучение  |
| val | 10% | Подбор гиперпараметров и порога |
| test | 10% | Проверка после подбора |
| late | 10% | Честная оценка в самом конце |

стратификация по target обязательна ( в каждой части будет ~8% дефолтов)

In [189]:
# RANDOM_STATE = 42
# Сначала отделяем 10% для late_test
train_val_test, data_late = train_test_split(
    data,
    test_size=0.1,  # 10% на late_test
    stratify=data['target'],
    random_state=RANDOM_STATE
)

# Из оставшихся 90% берем 70% для train (это ~77.8% от 90%)
# train_size = 0.7 / 0.9 ≈ 0.7778
data_train, val_test = train_test_split(
    train_val_test,
    train_size=0.7778,  # 70% от ВСЕХ данных
    stratify=train_val_test['target'],
    random_state=RANDOM_STATE
)

# Из оставшихся 20% делим поровну на val и test
data_val, data_test = train_test_split(
    val_test,
    test_size=0.5,  # 10% от ВСЕХ данных
    stratify=val_test['target'],
    random_state=RANDOM_STATE
)

In [190]:
# Сохраняем данные
data_train.to_csv('train.csv', index=False)
data_val.to_csv('val.csv', index=False)
data_test.to_csv('test.csv', index=False)
data_late.to_csv('late.csv', index=False)

In [191]:
# Бьём данные на X и y
data_train_sampled = resample(data_train, n_samples=50_000, replace=False, random_state=RANDOM_STATE)

X_train, y_train = split_xy(data_train_sampled)
X_val, y_val = split_xy(data_val)
X_test, y_test = split_xy(data_test)
X_late, y_late = split_xy(data_late)

In [192]:
# Encoding
categorical_columns = X_train.select_dtypes(include=['str']).columns

columns_for_ord_enc = ['type_of_occupation', 'type_of_organization']
columns_for_ohe =  [col for col in categorical_columns if (col not in columns_for_ord_enc)]

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ord_enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

def process_split(X, fit=False):
    num_part = X[num_cols]
    cat_parts = X[cat_cols]
    
    ohe_array = ohe.fit_transform(X[columns_for_ohe]) if fit else ohe.transform(X[columns_for_ohe])
    ohe_df = pd.DataFrame(ohe_array, columns=ohe.get_feature_names_out(columns_for_ohe), index=X.index)
    
    ord_array = ord_enc.fit_transform(X[columns_for_ord_enc]) if fit else ord_enc.transform(X[columns_for_ord_enc])
    ord_df = pd.DataFrame(ord_array, columns=ord_enc.get_feature_names_out(columns_for_ord_enc), index=X.index)
    return pd.concat([ohe_df, ord_df, num_part], axis=1)

X_train_enc = process_split(X_train, fit=True)
X_val_enc = process_split(X_val, fit=False)
X_test_enc = process_split(X_test, fit=False)
X_late_enc = process_split(X_late, fit=False)

In [193]:
# Scaling
scaler = StandardScaler()
X_train_enc_trans = scaler.fit_transform(X_train_enc)
X_val_enc_trans = scaler.transform(X_val_enc)
X_test_enc_trans = scaler.transform(X_test_enc)
X_late_enc_trans = scaler.transform(X_late_enc)

In [194]:
model = LogisticRegression(random_state=RANDOM_STATE)

model.fit(X_train_enc_trans, y_train)

probabilities = model.predict_proba(X_val_enc_trans)[:,1]
gini(y_val, probabilities)

0.4502710325652286